# KOPIS 수집본 검증과 불변 분석 스냅샷

수집이 멈춘 회차의 통합 CSV를 정상 API 원문과 대조해 분석 입력을 고정한다. 이 노트북은 **네트워크 요청을 하지 않는다**. 기존 원본·기준선 분석은 수정하지 않는다.

`reports/kopis_collection/latest.json`이 가리키는 종료 회차를 기본 대상으로 한다. 환경변수 `KOPIS_SNAPSHOT_RUN_ID`를 지정하면 해당 회차 보고서를 읽되, 현재 통합 CSV와 원문이 그 보고서의 건수와 맞아야 한다. 실행 중이거나 수집 파일이 바뀌면 저장을 거절한다. 분석용 CSV는 `dataset/analysis/kopis_snapshots/<run_id>/`, 매니페스트·검증 결과는 `reports/kopis_collection/<run_id>/`에 최초 한 번만 작성하고 재실행 시 동일 해시를 검증한다.

모든 검증이 끝나면 검증된 매니페스트를 `metadata/kopis_analysis_sources.json`에 원자적으로 반영한다. 이후 03번 노트북은 이 포인터가 선택한 불변 CSV를 읽는다. 수집 중이거나 검증에 실패하면 기존 분석 입력 선택은 유지한다.

이 노트북은 필요할 때만 사용하는 **수집 관리 도구**이며 최종 분석의 전체 실행에는 포함하지 않는다. 수집 및 스냅샷 등록은 공용 원자료와 `metadata/`에 기록된다. `jupyter-notebook/20261006/metadata/`에 고정한 분석 입력은 자동 교체하지 않는다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
from datetime import datetime, timezone, timedelta
import csv, hashlib, io, json, os
import xml.etree.ElementTree as ET
import pandas as pd
from IPython.display import display

ROOT = PROJECT_ROOT
REPORTS = project_path('reports/kopis_collection')
MERGED = project_path('dataset/analysis/kopis_collection')
RAW = project_path('dataset/raw/kopis')
ADDITIONS = project_path('dataset/raw/kopis_recollection')
KST = timezone(timedelta(hours=9))
def collector_active():
    lock_path = REPORTS / "collector.lock"
    if not lock_path.exists():
        return False
    with lock_path.open("r+b") as stream:
        acquired = False
        try:
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(stream.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            acquired = True
            return False
        except OSError:
            return True
        finally:
            if acquired:
                stream.seek(0)
                if os.name == "nt":
                    msvcrt.locking(stream.fileno(), msvcrt.LK_UNLCK, 1)
                else:
                    fcntl.flock(stream.fileno(), fcntl.LOCK_UN)

assert not collector_active(), "실행 중인 수집기가 있습니다. 종료 후 다시 실행하세요."
latest_path = REPORTS / "latest.json"
latest_blob_before = latest_path.read_bytes()
latest_before = project_json_loads(latest_blob_before)
assert latest_before["status"] != "running", "최신 회차가 수집 중입니다."
assert latest_before.get("finished_at_kst"), "최신 회차의 종료를 확인할 수 없습니다."
requested_run = os.environ.get("KOPIS_SNAPSHOT_RUN_ID", "")
report_path = REPORTS / (requested_run + ".json" if requested_run else "latest.json")
report_blob = report_path.read_bytes()
state = project_json_loads(report_blob)
assert state["status"] != "running", "진행 중 수집본은 고정하지 않습니다."
assert state.get("finished_at_kst"), "종료 시각이 없는 회차는 고정하지 않습니다."
run_id = state["run_id"]
assert Path(run_id).name == run_id and "/" not in run_id and "\\" not in run_id
DEST = REPORTS / run_id
DETAILS = project_path('dataset/analysis/kopis_snapshots') / run_id
SPECS = {"facility_details": ("facility", "mt10id"), "performance_details": ("performance", "mt20id")}

def digest(blob):
    return hashlib.sha256(blob).hexdigest()

def csv_rows(blob):
    return list(csv.DictReader(io.StringIO(blob.decode("utf-8-sig"))))

## 1. 통합 CSV와 공식 ID 검증

CSV를 바이트로 한 번 읽고 ID 누락·중복과 회차별 확보 건수를 확인한다. 시설 CSV의 자치구는 기존 공식 시설 목록에서 가져온 파생 열이다. 공연 상세의 `mt20id → mt10id` 연결만 공식 연결로 센다.

In [ ]:
tables, blobs, merged_paths, audit_rows = {}, {}, {}, []
for table_name, (kind, idkey) in SPECS.items():
    path = MERGED / f"{table_name}.csv"
    blob = path.read_bytes()
    rows = csv_rows(blob)
    ids = [row[idkey] for row in rows]
    assert all(ids) and len(ids) == len(set(ids)), f"{table_name}: ID 누락 또는 중복"
    assert len(ids) == state["final_counts"][kind], f"{table_name}: 보고서와 건수 불일치"
    tables[table_name] = {row[idkey]: row for row in rows}
    blobs[table_name], merged_paths[table_name] = blob, path
    audit_rows.append({"자료": table_name, "확보": len(ids), "대기": state["pending_counts"][kind],
                       "공식 시설 연결": sum(bool(r.get("mt10id")) for r in rows) if kind == "performance" else None})
display(pd.DataFrame(audit_rows))

해석: 확보 건수와 미수집 수는 사전에 지정한 수집 대상 목록을 기준으로 한다. 연구 후보 공연 상세를 모두 확보했더라도 서울 전체 공연 목록의 상세를 전부 받은 것은 아니다. 중단된 회차도 그 전에 저장된 정상 상세는 검증 후 사용할 수 있으며, 미제공 값을 0건 공연이나 0원 매출로 해석하지 않는다.

## 2. 정상 원문과 셀 값 대조

기존 원문과 추가 수집 회차에서 각 ID의 최신 정상 XML을 찾는다. API 응답의 공식 ID와 파일명을 대조하고 CSV의 원문 유래 값까지 비교한다. 원문마다 SHA-256, 크기, 상대 경로를 기록한다. 기존 원문은 읽기만 한다.

In [ ]:
raw_provenance = []
for table_name, (_, idkey) in SPECS.items():
    paths = list((RAW / table_name).glob("*.xml")) + sorted(ADDITIONS.glob(f"*/{table_name}/*.xml"))
    latest_paths = {path.stem: path for path in paths}
    assert set(latest_paths) == set(tables[table_name]), f"{table_name}: CSV와 원문 ID 집합 불일치"
    for record_id, path in sorted(latest_paths.items()):
        blob = path.read_bytes()
        root = ET.fromstring(blob)
        assert root.tag == "dbs" and len(root.findall("db")) == 1
        row = {e.tag: (e.text or "").strip() for e in root.find("db") if len(e) == 0}
        assert row.get(idkey) == record_id
        assert all(tables[table_name][record_id].get(k, "") == v for k, v in row.items()), f"{record_id}: 원문과 CSV 값 불일치"
        raw_provenance.append({"table": table_name, "id_key": idkey, "record_id": record_id,
                               "path": path.relative_to(ROOT).as_posix(),
                               "sha256": digest(blob), "bytes": len(blob)})
display(pd.DataFrame(raw_provenance).groupby("table").agg(원문수=("record_id", "count"), 원문바이트=("bytes", "sum")))

해석: 해시는 저장된 원문과 분석 입력의 일치 여부를 추적한다. 과거 원본의 최신성이나 시설 이전 여부를 보증하지 않는다. 2026년 재수집 좌표·시설 정보와 2023~2025년 관측기간의 차이는 남아 있다.

## 3. 범위와 좌표 누락 확인

기존 후보의 구역·반경 조건은 외식업종을 사용하지 않아 외식업 10종 확장에서도 유지한다. 후보 표의 시설명 연결과 상세 API의 공식 ID 연결을 구분한다. 좌표 존재·대한민국 범위 검사는 공간 결합 정확성 검증과 별개다.

In [ ]:
legacy_names = ["facilities_list_seoul.csv", "performance_detail_candidates.csv", "performances_list_seoul_2023_2025.csv"]
input_sources, legacy = [], {}
for name in legacy_names:
    path = project_path('dataset/processed/kopis') / name
    blob = path.read_bytes()
    legacy[name] = csv_rows(blob)
    input_sources.append({"path": path.relative_to(ROOT).as_posix(), "sha256": digest(blob), "bytes": len(blob)})
facility_list = legacy["facilities_list_seoul.csv"]
candidates = legacy["performance_detail_candidates.csv"]
target_facilities = {r["mt10id"] for r in facility_list if r["gugunnm"] in {"종로구", "마포구"}}
target_performances = {r["mt20id"] for r in candidates if "target_district" in r["candidate_reason"].split(";")}
facility_frame = pd.DataFrame(tables["facility_details"].values())
lon = pd.to_numeric(facility_frame["lo"], errors="coerce")
lat = pd.to_numeric(facility_frame["la"], errors="coerce")
quality = {"target_facilities": len(target_facilities),
           "target_facility_pending": len(target_facilities - tables["facility_details"].keys()),
           "target_candidate_performances": len(target_performances),
           "target_candidate_performance_pending": len(target_performances - tables["performance_details"].keys()),
           "official_performance_facility_links": sum(bool(r.get("mt10id")) for r in tables["performance_details"].values()),
           "coordinates_missing_or_invalid": int((~(lon.between(124, 132) & lat.between(33, 39))).sum())}
display(pd.DataFrame([quality]).T.rename(columns={0: "건수"}))

# 완료 여부는 보고서 건수뿐 아니라 실제 대상 ID 집합으로 검증한다.
collection_targets = {
    "facility": {row["mt10id"] for row in facility_list},
    "performance": {row["mt20id"] for row in candidates},
}
coverage_rows = []
for table_name, (kind, idkey) in SPECS.items():
    expected = collection_targets[kind]
    acquired = set(tables[table_name])
    pending = expected - acquired
    assert len(expected) == state["target_counts"][kind], f"{kind}: 대상 건수 불일치"
    assert len(pending) == state["pending_counts"][kind], f"{kind}: 실제 미수집 ID 불일치"
    coverage_rows.append({"자료": kind, "수집 대상": len(expected),
                          "대상 중 확보": len(expected & acquired), "대상 중 미수집": len(pending)})
if state["status"] == "completed_pending_queue":
    assert all(row["대상 중 미수집"] == 0 for row in coverage_rows)
display(pd.DataFrame(coverage_rows))


해석: 수집 완료는 위 후보 ID 집합의 미수집 수가 0이라는 뜻이다. 후보는 종로·마포 또는 주변 800m와 당시 좌표 미확보 시설을 포함한 기존 목록이다. 최종 연구 범위는 현재 확보한 공식 시설 ID와 좌표로 다시 판정한다. 후보 밖 공연은 여전히 상세가 없을 수 있으며, 현재 경계의 과거 적용과 시설명 임시 연결의 한계는 후속 분석에서 따로 표시한다.

## 4. 불변 스냅샷과 매니페스트 저장

읽기 전과 저장 직전에 OS 파일 잠금으로 수집기 실행 여부를 확인하고, 최신 회차의 상태·run_id·종료 시각·보고서 바이트가 같은지 확인한다. 통합 CSV·보고서·원문의 해시도 다시 대조한다. 각 파일은 배타적 생성 모드로 최초 저장하며, 기존 스냅샷이면 바이트가 동일할 때만 재사용한다. 매니페스트를 마지막에 저장하므로 `validated: true`인 완성본만 분석에서 읽는다.

In [ ]:
assert not collector_active(), "검증 중 수집기가 실행되었습니다."
latest_blob_after = latest_path.read_bytes()
latest_after = project_json_loads(latest_blob_after)
assert latest_after["status"] != "running"
assert (latest_after["run_id"], latest_after["finished_at_kst"]) == (latest_before["run_id"], latest_before["finished_at_kst"])
assert latest_blob_after == latest_blob_before, "검증 중 최신 수집 회차 변경"
assert report_path.read_bytes() == report_blob, "검증 중 수집 보고서 변경"
for table_name, path in merged_paths.items():
    assert path.read_bytes() == blobs[table_name], "검증 중 통합 CSV 변경"
for source in raw_provenance + input_sources:
    assert digest((project_path(source["path"])).read_bytes()) == source["sha256"], "검증 중 원본 변경"
DEST.mkdir(parents=True, exist_ok=True)
DETAILS.mkdir(parents=True, exist_ok=True)

def preserve_bytes(path, blob):
    if path.exists():
        assert path.read_bytes() == blob, f"기존 불변 스냅샷과 불일치: {path.name}"
    else:
        with path.open("xb") as stream:
            stream.write(blob)

files = {}
for table_name, (_, idkey) in SPECS.items():
    path = DETAILS / f"{table_name}.csv"
    preserve_bytes(path, blobs[table_name])
    files[table_name] = {"path": path.relative_to(ROOT).as_posix(), "sha256": digest(blobs[table_name]),
                         "rows": len(tables[table_name]), "id_key": idkey}
provenance_blob = pd.DataFrame(raw_provenance).to_csv(index=False).encode("utf-8-sig")
preserve_bytes(DEST / "raw_source_manifest.csv", provenance_blob)
preserve_bytes(DEST / "collection_run.json", report_blob)

In [ ]:
manifest_path = DEST / "snapshot_manifest.json"
manifest = {"run_id": run_id, "status": state["status"], "validated": True,
            "validated_at_kst": datetime.now(KST).isoformat(), "files": files,
            "collection_finished_at_kst": state["finished_at_kst"],
            "collection_report": {"path": (DEST / "collection_run.json").relative_to(ROOT).as_posix(), "sha256": digest(report_blob)},
            "raw_sources": {"path": (DEST / "raw_source_manifest.csv").relative_to(ROOT).as_posix(),
                            "sha256": digest(provenance_blob), "rows": len(raw_provenance)},
            "input_sources": input_sources, "quality": quality,
            "pending_counts": state["pending_counts"],
            "limitations": ["시설명 기반 후보는 공식 연결이 아님", "미수집 상세는 0으로 보완하지 않음",
                            "현재 시설 좌표와 경계의 과거 유효성 미검증", "기존 서울 목록과 후보 목록은 재수집하지 않음"]}
if manifest_path.exists():
    saved_manifest = project_json_loads(manifest_path.read_text(encoding="utf-8"))
    assert {k: v for k, v in saved_manifest.items() if k != "validated_at_kst"} == {k: v for k, v in manifest.items() if k != "validated_at_kst"}
    manifest = saved_manifest
else:
    preserve_bytes(manifest_path, (json.dumps(manifest, ensure_ascii=False, indent=2) + "\n").encode("utf-8"))
display(pd.DataFrame(files).T[["rows", "id_key", "path", "sha256"]])
print("검증 완료:", manifest_path.relative_to(ROOT).as_posix())

## 5. 후속 분석 입력 선택

검증된 매니페스트 원본 바이트를 `metadata/kopis_analysis_sources.json`에 복사한다. 저장 직전에 수집 상태와 CSV 해시를 다시 확인하며, 같은 바이트라면 쓰기를 생략한다. 중간 파일은 같은 폴더에 쓰고 원자적으로 교체하므로 분석이 부분 기록을 읽지 않는다.

In [ ]:
assert not collector_active(), "수집 중에는 분석 입력 포인터를 갱신하지 않습니다."
assert latest_path.read_bytes() == latest_blob_before, "검증 후 최신 수집 회차 변경"
manifest_blob = manifest_path.read_bytes()
selected_manifest = project_json_loads(manifest_blob)
assert selected_manifest["validated"] is True and selected_manifest["run_id"] == run_id
for table_name, entry in selected_manifest["files"].items():
    assert digest((project_path(entry["path"])).read_bytes()) == entry["sha256"], "고정 CSV 해시 불일치"
    assert merged_paths[table_name].read_bytes() == blobs[table_name], "검증 후 통합 CSV 변경"

pointer_path = project_path('metadata/kopis_analysis_sources.json')
if pointer_path.exists() and pointer_path.read_bytes() == manifest_blob:
    pointer_status = "동일한 분석 입력 유지: 쓰기 생략"
else:
    pointer_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_pointer = pointer_path.with_name(pointer_path.name + f".{os.getpid()}.tmp")
    try:
        temporary_pointer.write_bytes(manifest_blob)
        assert not collector_active() and latest_path.read_bytes() == latest_blob_before
        temporary_pointer.replace(pointer_path)
    finally:
        temporary_pointer.unlink(missing_ok=True)
    pointer_status = "검증된 분석 입력 선택 갱신"
print(pointer_status, "|", run_id)
print("분석 포인터:", pointer_path.relative_to(ROOT).as_posix())

검증된 새 수집본은 공용 입력 명세에 등록된다. 현재 최종 분석은 `jupyter-notebook/20261006/metadata/`의 고정 명세를 계속 사용한다. 새 회차를 분석에 채택하려면 최종 입력 명세를 별도 갱신하고 `jupyter-notebook/20261006/00_전체실행.ipynb`를 실행한다.